# 1. 우선 BERT기반 언어모델로 대화해보고 그다음은 GPT로 대화해보자
- 지금 내가 하는건 단순 API를 받아오는것이 아니다 (내 컴퓨터에 가중치와 코드파일이 있는지 확인하고 HuggigFace 서버로 HTTPS 다운로드를 보내는것이다)
- 모델 가져와서 실행하면 그때부터는 전부 로컬에서 실행되는것이지 HuggingFace서버에서 계산하지 않는다.
- 인터넷 없이도 실행가능하다는것은 내 컴퓨터에 언어모델 가중치와 forward가능 구조 들어와있는것
- 클라우드API와 차이는 예를들어 OpenAI API를 이용한다고 했을때 내 컴퓨터에서 명령어를 서버로 보내면 서버에서 계산을하고 결과를 반환하지만 지금 내가하는 허깅페이스 이용하여 모델 다운받는것은 내 컴퓨터에서 로컬로 모델 돌리고 계산해 명령어 자체적으로 수행하는것이다.

In [3]:
!pip install transformers

# HuggingFace의 transformers 라이브러리 코드 자체를 내 컴퓨터에 설치(모델 구조 코드, 다운로드 기능, 토크나이저 코드, config로딩 기능을 포함한 파이썬 패키지)
# 즉 “모델을 다룰 수 있는 도구” 설치하는것

  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 1.4 MB/s  0:00:07a 0:00:0136m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 957.3 kB/s  0:00:00935.2 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 1.1 MB/s  0:00:03m 1.1 MB/s eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 2.3 MB/s  0:00:01m 2.2 MB/s eta 0:00:01
Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [transformers] 14/15 [transformers]ub]


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

/Users/pc/miniforge3/envs/research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch
from transformers import BertModel, AutoTokenizer

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = BertModel.from_pretrained("google-bert/bert-base-uncased")

model.to(device)

inputs = tokenizer("Hello world", return_tensors="pt").to(device)

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

# 이 코드만 실행했을때 from transformers import BertModel을 해석할수있는 라이브러리가 없어서 오류가 나왔었다

Loading weights: 100%|█| 199/199 [00:00<00:00, 1424.47it/s, Materializing param=
BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 4, 768])


In [9]:
'''
내가 위 코드들을 실행하며 다운로드한 파일들은 바로 주피터 노트북환경 이 노트북 근처에 다운로드되어 보이지 않은 이유는
기본적으로 macOS에서 HuggingFace 모델은 여기(/Users/너의아이디/.cache/huggingface/)에 저장 되기때문이다 => 사용자 홈 디렉토리의 숨김 폴더(.cache)에 저장
<이유>
왜 캐시 폴더에 저장하나?
이유는:
여러 프로젝트에서 공유 가능
매번 다운로드 안 해도 됨
중복 저장 방지
즉, 시스템 전체에서 공용으로 쓰는 모델 저장소 (그러니 어디있는지 알고 폴더의 이름을 함부로 바꾸지 말자 다음에 못찾고 다시 다운로드할수도있다)
'''

import os
print(os.path.expanduser("~/.cache/huggingface")) # 위 파일들 어디에 다운로드되는지 경로 출력해주는코드

'''
출력결과 /Users/pc/.cache/huggingface인데 여기서 cache앞에 .이 찍혀있다 이런것은 보통 숨김폴더(hidden folder)로 그냥 Finder에서는 안보인다
<그러나 보는법>
Finder 열고 아무 폴더에서:
⌘ + Shift + . => pc폴더에서 내가 확인해봤는데 .cache .vscode .keras . android .ssh등 많은 숨김폴더 있다
(커맨드 + 쉬프트 + 마침표)
누르면 숨김 파일이 보인다.
다시 누르면 숨김 해제.
숨김폴더로 저장하는이유는 시스템 캐시이고 중요하니 사용자 실수방지로 걸드리지말라고 프로젝트 폴더와 분리
'''

/Users/pc/.cache/huggingface
